In [1]:
import pandas as pd

excel_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
df = pd.read_excel(excel_path, engine='openpyxl')

df

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,104.131.6.219,2026-02-18,Address,95,2,0.994795,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",Incident:546888,NaN,260,328,315,medium,[2026-02-18] Severity: medium. VT score: 6. To...
1,104.18.32.191,2026-02-17,Address,286,3,0.984329,1,0,"CDC, CMS, DHA, HHS, HRSA, IHS, NIH, VA",Incident:288387,NaN,70,233,70,low,[2026-02-18] Severity: low. VT score: 0. Top d...
2,104.21.79.144,2026-02-17,Address,14,1,0.999233,0,0,"CMS, NIH, VA",Incident:6755399477000935,NaN,70,174,130,low,[2026-02-18] Severity: low. VT score: 0. Top d...
3,138.197.101.95,2026-02-15,Address,79,3,0.995671,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",Incident:546888,NaN,210,383,303,medium,[2026-02-18] Severity: medium. VT score: 4. To...
4,14.103.127.80,2026-02-17,Address,72,4,0.996055,0,0,"CMS, FDA, HHS, HRSA, IHS, OS",Incident:546441,NaN,550,553,310,medium,[2026-02-18] Severity: medium. VT score: 5. To...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2156,5.79.110.170,2025-12-23,Address,6,0,0.999671,1,0,"CDC, CMS, HHS",Incident:6755399448003491;Incident:529312,NaN,170,363,45,low,Severity: low. VT score: 0. Top drivers: Incid...
2157,146.70.246.116,2026-01-19,Address,3,0,0.999836,1,0,VA,Incident:6755399478001848,NaN,180,368,41,low,Severity: low. VT score: 0. Top drivers: Incid...
2158,198.199.103.243,2026-01-16,Address,1,0,0.999945,1,0,VA,Incident:6755399478001848,NaN,180,368,41,low,Severity: low. VT score: 0. Top drivers: Incid...
2159,rosesci.com,2026-01-08,Host,17,0,0.999068,0,0,NIH,NaN,NaN,170,351,30,low,Severity: low. Could not pull VT score to calc...


In [27]:
import glob
import os

csv_dir = r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports"
csv_pattern = "full_daily_report_*.csv"
csv_paths = sorted(glob.glob(f"{csv_dir}\\{csv_pattern}"))

print(f"Found {len(csv_paths)} CSV files")

dfs = []
loaded_files = []
skipped_files = []

for path in csv_paths:
    try:
        # Check if file exists and has content
        if not os.path.exists(path):
            skipped_files.append((path, "File not found"))
            continue
            
        file_size = os.path.getsize(path)
        if file_size == 0:
            skipped_files.append((path, "Empty file"))
            continue
        
        # Read CSV
        df_temp = pd.read_csv(path)
        
        # Check if DataFrame is empty
        if df_temp.empty:
            skipped_files.append((path, "No data rows"))
            continue
        
        dfs.append(df_temp)
        loaded_files.append((path, len(df_temp)))
        
    except Exception as e:
        skipped_files.append((path, f"Error: {str(e)}"))
        continue

print(f"\nLoaded {len(loaded_files)} files successfully")
print(f"Skipped {len(skipped_files)} files")

if skipped_files:
    print("\nSkipped files:")
    for path, reason in skipped_files[:10]:  # Show first 10
        print(f"  {os.path.basename(path)}: {reason}")
    if len(skipped_files) > 10:
        print(f"  ... and {len(skipped_files) - 10} more")

if dfs:
    # Use sort=False to preserve column order, handle mismatches
    df_daily_reports = pd.concat(dfs, ignore_index=True, sort=False)
    print(f"\nTotal rows loaded: {len(df_daily_reports)}")
    print(f"Total columns: {len(df_daily_reports.columns)}")
    
    # Fill missing values with "unknown" for object/string columns
    for c in df_daily_reports.select_dtypes(include=['object']).columns:
        df_daily_reports[c] = df_daily_reports[c].fillna('unknown')
    
    # Fill missing numeric values with median per indicator
    import numpy as np
    numeric_cols = df_daily_reports.select_dtypes(include=[np.number]).columns.tolist()
    if 'Indicator' in df_daily_reports.columns and numeric_cols:
        for col in numeric_cols:
            # Calculate median per indicator group
            median_by_indicator = df_daily_reports.groupby('Indicator')[col].transform('median')
            # Fill missing values with the median for that indicator
            df_daily_reports[col] = df_daily_reports[col].fillna(median_by_indicator)
            # If indicator group has all NaN, fill with overall median
            df_daily_reports[col] = df_daily_reports[col].fillna(df_daily_reports[col].median())
    
else:
    print("\nWARNING: No data was loaded from any files!")
    df_daily_reports = pd.DataFrame()

df_daily_reports

Found 227 CSV files

Loaded 227 files successfully
Skipped 0 files

Total rows loaded: 2174379
Total columns: 17


,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,Partner,FileDate,ensemble_45d,Confidence: 45-Day,Frequency (1d),Probability: 1-Day,Confidence: 1-Day
0,102.129.153.158,0,0.0,1.0,6.88%,7-Day: Low confidence,13.68%,14-Day: Low confidence,68.66%,30-Day: Possibly active,CDC,20250616,unknown,unknown,0.0,unknown,unknown
1,102.129.153.43,0,0.0,0.0,0.38%,7-Day: Low confidence,0.93%,14-Day: Low confidence,18.11%,30-Day: Low confidence,CDC,20250616,unknown,unknown,0.0,unknown,unknown
2,102.129.153.71,0,0.0,2.0,12.39%,7-Day: Low confidence,24.8%,14-Day: Low confidence,86.71%,30-Day: Highly likely,CDC,20250616,unknown,unknown,0.0,unknown,unknown
3,102.165.16.161,0,0.0,0.0,0.01%,7-Day: Low confidence,0.01%,14-Day: Low confidence,0.17%,30-Day: Low confidence,CDC,20250616,unknown,unknown,0.0,unknown,unknown
4,103.133.107.28,0,0.0,1.0,9.14%,7-Day: Low confidence,59.14%,14-Day: Possibly active,73.35%,30-Day: Possibly active,CDC,20250616,unknown,unknown,0.0,unknown,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2174374,warlockstallioniso.com,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.0%,30-Day: Low confidence,VA,20260219,0.25%,45-Day: Low confidence,0.0,0.02%,1-Day: Low confidence
2174375,www.citiquartz.com,0,0.0,1.0,4.75%,7-Day: Low confidence,8.74%,14-Day: Low confidence,58.21%,30-Day: Possibly active,VA,20260219,46.27%,45-Day: Possibly active,0.0,0.84%,1-Day: Low confidence
2174376,www.deepseek.com,0,2.0,21.0,97.9%,7-Day: Highly likely,99.27%,14-Day: Highly likely,100.0%,30-Day: Highly likely,VA,20260219,92.52%,45-Day: Highly likely,0.0,35.86%,1-Day: Low confidence
2174377,www.deepseek.com.cdn.cloudflare.net,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.0%,30-Day: Low confidence,VA,20260219,0.13%,45-Day: Low confidence,0.0,0.01%,1-Day: Low confidence


In [28]:
import numpy as np

# Aggregate by Indicator and take averages for everything else
def parse_pct(s):
    """Parse '85%' or '85.2%' to float."""
    if pd.isna(s) or s == '':
        return np.nan
    try:
        return float(str(s).strip().rstrip('%'))
    except ValueError:
        return np.nan

# Drop file-related columns (e.g. FileDate) before aggregating
df_daily_reports = df_daily_reports.drop(columns=['FileDate'], errors='ignore')

numeric_cols = df_daily_reports.select_dtypes(include=[np.number]).columns.tolist()
# Separate Probability (percentages) from Confidence (descriptive strings)
prob_cols = [c for c in df_daily_reports.columns if c != 'Indicator' and ('Probability' in c or c == 'ensemble_45d')]
conf_cols = [c for c in df_daily_reports.columns if c != 'Indicator' and 'Confidence' in c]
obj_cols = [c for c in df_daily_reports.columns if c != 'Indicator' and c not in numeric_cols and c not in prob_cols and c not in conf_cols]

agg_dict = {c: 'mean' for c in numeric_cols}
agg_dict.update({c: 'first' for c in obj_cols})
# Confidence columns use mode (most frequent label)
def make_mode_agg():
    def mode_agg(x):
        mode_vals = x.mode()
        if len(mode_vals) > 0:
            return mode_vals[0]
        elif len(x) > 0:
            return x.iloc[0]
        else:
            return 'unknown'
    return mode_agg

for c in conf_cols:
    agg_dict[c] = make_mode_agg()
# Only parse Probability columns as percentages
def make_pct_agg():
    def pct_agg(x):
        vals = x.apply(parse_pct)
        if vals.notna().any():
            return f"{vals.mean():.2f}%"
        else:
            return "unknown"
    return pct_agg

for c in prob_cols:
    agg_dict[c] = make_pct_agg()

df_daily_agg = df_daily_reports.groupby('Indicator').agg(agg_dict).reset_index()

# Round numeric columns to 2 decimal places
for c in df_daily_agg.select_dtypes(include=[np.number]).columns:
    df_daily_agg[c] = df_daily_agg[c].round(2)

# Rename columns to reflect aggregation method
rename_dict = {}
# Numeric columns use mean (average)
for c in numeric_cols:
    rename_dict[c] = f"{c} (Average)"
# Probability columns use mean (average) of percentages
for c in prob_cols:
    if c == 'ensemble_45d':
        rename_dict[c] = "Probability: 45-Day (Average)"
    else:
        rename_dict[c] = f"{c} (Average)"
# Object columns use first
for c in obj_cols:
    rename_dict[c] = f"{c} (First)"
# Confidence columns use mode
for c in conf_cols:
    rename_dict[c] = f"{c} (Mode)"

df_daily_agg = df_daily_agg.rename(columns=rename_dict)

del df_daily_reports  # drop raw file data
df_daily_agg

,Indicator,Observed Today (Average),Frequency (7d) (Average),Frequency (30d) (Average),Frequency (1d) (Average),Partner (First),Confidence: 7-Day (Mode),Confidence: 14-Day (Mode),Confidence: 30-Day (Mode),Confidence: 45-Day (Mode),Confidence: 1-Day (Mode),Probability: 7-Day (Average),Probability: 14-Day (Average),Probability: 30-Day (Average),Probability: 45-Day (Average),Probability: 1-Day (Average)
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0.00,0.00,1.11,0.00,VA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,1-Day: Low confidence,21.81%,44.00%,63.79%,62.58%,0.74%
1,1-you.njalla.no,0.01,0.46,1.91,0.01,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,unknown,unknown,26.03%,43.72%,64.62%,49.98%,10.83%
2,1.192.18.4,0.01,0.21,0.95,0.00,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Low confidence,unknown,15.94%,29.39%,46.53%,43.33%,0.01%
3,1.204.166.3,0.00,0.12,0.71,0.00,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,unknown,12.10%,24.36%,48.52%,41.00%,0.08%
4,1.234.83.26,0.44,4.29,14.75,0.44,CMS,7-Day: Highly likely,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,1-Day: Low confidence,91.87%,94.12%,98.29%,77.67%,48.12%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5627,ymath.yonsei.ac.kr,0.00,0.00,0.03,0.00,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Low confidence,1-Day: Low confidence,0.17%,1.02%,2.64%,12.94%,0.02%
5628,yotpo-static.com,0.00,0.02,0.08,0.00,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Low confidence,1-Day: Low confidence,1.39%,1.69%,5.26%,13.81%,0.04%
5629,yourpensionmeeting.com,0.32,2.76,10.96,0.17,VA,7-Day: Low confidence,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,unknown,54.49%,65.18%,78.31%,64.37%,28.10%
5630,zerocap.com,0.00,0.00,0.00,0.00,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,unknown,0.00%,0.00%,0.00%,unknown,unknown
